# Exploratory Data Analysis (EDA)

**Project**: WGU D610 Capstone - Serverless FHIR Pipeline Performance Study  
**Author**: Robert Miller  
**Date**: November 2025  
**Purpose**: Understand data distributions, relationships, and quality before formal hypothesis testing

---

## Overview

This notebook performs comprehensive exploratory data analysis on the cleaned dataset from Phase 3. The analysis includes:

1. **Data Loading & Validation**: Verify data quality and structure
2. **Univariate Analysis**: Examine distributions of key variables
3. **Bivariate Analysis**: Explore relationships between variables
4. **Stratified Analysis**: Compare subgroups (rural/urban, data sources)
5. **Missing Data**: Assess completeness and patterns
6. **Normality Assessment**: Check assumptions for statistical tests

**Expected Output**: 15-20 visualizations saved to `reports/figures/eda/`

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(12345)

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

# Create output directory for figures
output_dir = Path('reports/figures/eda')
output_dir.mkdir(parents=True, exist_ok=True)

print("✓ Libraries imported and configuration complete")
print(f"✓ Output directory: {output_dir}")

## 2. Data Loading and Validation

In [ ]:
# Load processed data from Phase 3
data_path = Path('data/processed/combined_data.parquet')

if not data_path.exists():
    raise FileNotFoundError(
        f"Processed data not found at {data_path}. "
        "Please run Phase 3 data cleaning first."
    )

df = pd.read_parquet(data_path)

print(f"✓ Loaded dataset with {len(df):,} records")
print(f"✓ Columns: {len(df.columns)}")
print(f"\nDataset shape: {df.shape}")

In [ ]:
# Display basic information
print("\n=== Dataset Info ===")
df.info()

In [ ]:
# Preview first few records
print("\n=== Sample Records ===")
df.head(10)

In [ ]:
# Validation checks
print("\n=== Data Validation Checks ===")

# 1. Sample size
min_sample = 800
assert df.shape[0] >= min_sample, f"Insufficient sample size: {df.shape[0]} < {min_sample}"
print(f"✓ Sample size: {df.shape[0]:,} (≥{min_sample} required for 80% power)")

# 2. Required columns
required_cols = [
    'latency_sec',           # Primary DV
    'validation_pass_bool',  # Secondary DV
    'data_source',           # Primary IV
    'record_complexity',     # IV
    'file_size_kb',          # Control variable
]

missing_cols = set(required_cols) - set(df.columns)
assert len(missing_cols) == 0, f"Missing required columns: {missing_cols}"
print(f"✓ All {len(required_cols)} required columns present")

# 3. No missing outcomes
assert df['latency_sec'].notna().all(), "Missing values in latency_sec"
assert df['validation_pass_bool'].notna().all(), "Missing values in validation_pass_bool"
print("✓ No missing values in outcome variables")

# 4. Data types
assert pd.api.types.is_numeric_dtype(df['latency_sec']), "latency_sec not numeric"
assert pd.api.types.is_bool_dtype(df['validation_pass_bool']) or df['validation_pass_bool'].dtype == 'int64', "validation_pass_bool not boolean"
print("✓ Data types correct")

# 5. Value ranges
assert (df['latency_sec'] > 0).all(), "Non-positive latency values found"
assert (df['latency_sec'] < 600).all(), "Implausible latency values (>10 min)"
assert (df['file_size_kb'] >= 0).all(), "Negative file sizes found"
print("✓ Value ranges valid")

print("\n✓ All validation checks passed!")

## 3. Univariate Analysis

Examine the distribution of individual variables.

### 3.1 Primary Outcome: Latency

In [ ]:
# Summary statistics for latency
print("\n=== Latency Summary Statistics ===")
print(df['latency_sec'].describe())
print(f"\nMedian (P50): {df['latency_sec'].median():.2f} seconds")
print(f"P95: {df['latency_sec'].quantile(0.95):.2f} seconds")
print(f"P99: {df['latency_sec'].quantile(0.99):.2f} seconds")

In [ ]:
# Visualization 1: Latency distribution (histogram + KDE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Histogram
axes[0].hist(df['latency_sec'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['latency_sec'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["latency_sec"].median():.1f}s')
axes[0].axvline(df['latency_sec'].mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {df["latency_sec"].mean():.1f}s')
axes[0].set_xlabel('Latency (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Automated Pipeline Latency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel B: KDE with rug plot
sns.kdeplot(data=df, x='latency_sec', ax=axes[1], fill=True, color='steelblue')
sns.rugplot(data=df.sample(min(500, len(df)), random_state=12345), x='latency_sec', ax=axes[1], alpha=0.3)
axes[1].axvline(30, color='green', linestyle=':', linewidth=2, label='P50 Threshold (30s)')
axes[1].axvline(120, color='orange', linestyle=':', linewidth=2, label='P95 Threshold (120s)')
axes[1].set_xlabel('Latency (seconds)')
axes[1].set_ylabel('Density')
axes[1].set_title('Kernel Density Estimate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'latency_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: latency_distribution.png")

### 3.2 File Size Distribution

In [ ]:
# Summary statistics for file size
print("\n=== File Size Summary Statistics ===")
print(df['file_size_kb'].describe())

In [ ]:
# Visualization 2: File size distribution (linear and log scales)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Linear scale
axes[0].hist(df['file_size_kb'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0].set_xlabel('File Size (KB)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('File Size Distribution (Linear Scale)')
axes[0].grid(True, alpha=0.3)

# Panel B: Log scale (if data is skewed)
axes[1].hist(np.log10(df['file_size_kb'] + 1), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Log10(File Size + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('File Size Distribution (Log Scale)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'file_size_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: file_size_distribution.png")

### 3.3 Categorical Variables

In [ ]:
# Data source distribution
print("\n=== Data Source Distribution ===")
source_counts = df['data_source'].value_counts()
print(source_counts)
print(f"\nTotal sources: {df['data_source'].nunique()}")

In [ ]:
# Visualization 3: Data source bar chart
fig, ax = plt.subplots(figsize=(10, 6))
source_counts.plot(kind='bar', ax=ax, color=sns.color_palette("colorblind", len(source_counts)))
ax.set_xlabel('Data Source')
ax.set_ylabel('Number of Records')
ax.set_title('Distribution of Records by Data Source')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Add percentage labels
total = source_counts.sum()
for i, v in enumerate(source_counts):
    ax.text(i, v + (total * 0.02), f'{v:,}\n({v/total*100:.1f}%)', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(output_dir / 'data_source_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: data_source_distribution.png")

In [ ]:
# Validation pass rate
pass_rate = df['validation_pass_bool'].mean() * 100
print(f"\n=== Validation Results ===")
print(f"Overall Pass Rate: {pass_rate:.2f}%")
print(f"Total Records: {len(df):,}")
print(f"Passed: {df['validation_pass_bool'].sum():,}")
print(f"Failed: {(~df['validation_pass_bool']).sum():,}")

In [ ]:
# Visualization 4: Validation pie chart
fig, ax = plt.subplots(figsize=(8, 8))
validation_counts = df['validation_pass_bool'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green for pass, red for fail
labels = ['Pass', 'Fail']

ax.pie(validation_counts, labels=labels, autopct='%1.1f%%', 
       startangle=90, colors=colors, textprops={'fontsize': 14, 'weight': 'bold'})
ax.set_title(f'Validation Results (n={len(df):,})', fontsize=16, weight='bold')

plt.tight_layout()
plt.savefig(output_dir / 'validation_pie_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: validation_pie_chart.png")

## 4. Bivariate Analysis

Explore relationships between variables.

### 4.1 Latency by Data Source

In [ ]:
# Summary statistics by data source
print("\n=== Latency by Data Source ===")
latency_by_source = df.groupby('data_source')['latency_sec'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(latency_by_source.round(2))

In [ ]:
# Visualization 5: Boxplot by data source
fig, ax = plt.subplots(figsize=(12, 6))
df.boxplot(column='latency_sec', by='data_source', ax=ax, grid=False)
ax.set_xlabel('Data Source', fontsize=12)
ax.set_ylabel('Latency (seconds)', fontsize=12)
ax.set_title('Latency Distribution by Data Source', fontsize=14, weight='bold')
plt.suptitle('')  # Remove auto-title
ax.axhline(30, color='green', linestyle=':', linewidth=2, label='P50 Threshold (30s)', alpha=0.7)
ax.axhline(120, color='orange', linestyle=':', linewidth=2, label='P95 Threshold (120s)', alpha=0.7)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'latency_by_source_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: latency_by_source_boxplot.png")

In [ ]:
# Visualization 6: Violin plot by data source
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df, x='data_source', y='latency_sec', ax=ax, palette="colorblind")
ax.set_xlabel('Data Source', fontsize=12)
ax.set_ylabel('Latency (seconds)', fontsize=12)
ax.set_title('Latency Distribution by Data Source (Violin Plot)', fontsize=14, weight='bold')
ax.axhline(30, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax.axhline(120, color='orange', linestyle=':', linewidth=2, alpha=0.7)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'latency_by_source_violin.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: latency_by_source_violin.png")

### 4.2 Validation by Record Complexity

In [ ]:
# Validation pass rate by complexity
print("\n=== Validation Pass Rate by Record Complexity ===")
validation_by_complexity = df.groupby('record_complexity')['validation_pass_bool'].agg(['sum', 'count'])
validation_by_complexity['pass_rate'] = validation_by_complexity['sum'] / validation_by_complexity['count'] * 100
print(validation_by_complexity.round(2))

In [ ]:
# Visualization 7: Validation pass rate by complexity
fig, ax = plt.subplots(figsize=(10, 6))
validation_by_complexity['pass_rate'].plot(kind='bar', ax=ax, color=sns.color_palette("colorblind", len(validation_by_complexity)))
ax.set_xlabel('Record Complexity', fontsize=12)
ax.set_ylabel('Validation Pass Rate (%)', fontsize=12)
ax.set_title('Validation Pass Rate by Record Complexity', fontsize=14, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.axhline(95, color='red', linestyle='--', linewidth=2, label='Target (95%)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (idx, row) in enumerate(validation_by_complexity.iterrows()):
    ax.text(i, row['pass_rate'] + 1, f"{row['pass_rate']:.1f}%\n(n={int(row['count'])})", 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(output_dir / 'validation_by_complexity.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: validation_by_complexity.png")

### 4.3 Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['latency_sec', 'file_size_kb', 'validation_pass_bool']
# Add any other numeric variables if available

# Correlation matrix
corr_matrix = df[numeric_cols].corr()
print("\n=== Correlation Matrix ===")
print(corr_matrix.round(3))

In [ ]:
# Visualization 8: Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Correlation Matrix of Continuous Variables', fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig(output_dir / 'correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: correlation_matrix.png")

### 4.4 Scatter Plots

In [ ]:
# Visualization 9: File size vs latency scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

# Create numeric encoding for data sources
source_categories = df['data_source'].astype('category')
scatter = ax.scatter(df['file_size_kb'], df['latency_sec'], 
                    c=source_categories.cat.codes, 
                    alpha=0.6, s=30, cmap='tab10')

ax.set_xlabel('File Size (KB)', fontsize=12)
ax.set_ylabel('Latency (seconds)', fontsize=12)
ax.set_title('Latency vs File Size (colored by Data Source)', fontsize=14, weight='bold')

# Add trend line
if len(df) > 0:
    z = np.polyfit(df['file_size_kb'], df['latency_sec'], 1)
    p = np.poly1d(z)
    sorted_x = df['file_size_kb'].sort_values()
    ax.plot(sorted_x, p(sorted_x), 
            "r--", alpha=0.8, linewidth=2, label=f'Trend: y={z[0]:.4f}x+{z[1]:.2f}')

# Add legend for data sources
unique_sources = df['data_source'].unique()
handles = [plt.Line2D([0], [0], marker='o', color='w', 
                     markerfacecolor=plt.cm.tab10(i), markersize=8, alpha=0.6) 
          for i in range(len(unique_sources))]
ax.legend(handles, unique_sources, title='Data Source', loc='best')

ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'latency_vs_file_size_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: latency_vs_file_size_scatter.png")

## 5. Stratified Analysis

Compare performance across subgroups.

### 5.1 Rural vs Urban (if available)

In [ ]:
# Check if rural_urban column exists
if 'rural_urban' in df.columns:
    print("\n=== Rural vs Urban Comparison ===")
    rural_urban_stats = df.groupby('rural_urban')['latency_sec'].agg(['count', 'mean', 'median', 'std'])
    print(rural_urban_stats.round(2))
    
    # Visualization 10: Split violin plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.violinplot(data=df, x='rural_urban', y='latency_sec', 
                   hue='data_source', ax=ax, palette="colorblind", split=False)
    ax.set_xlabel('Geographic Classification', fontsize=12)
    ax.set_ylabel('Latency (seconds)', fontsize=12)
    ax.set_title('Latency Distribution: Rural vs Urban by Data Source', fontsize=14, weight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'rural_urban_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved: rural_urban_comparison.png")
else:
    print("\n⚠ rural_urban column not found, skipping this analysis")

### 5.2 Health District Analysis (if available)

In [ ]:
# Check if health_district column exists and has multiple values
if 'health_district' in df.columns and df['health_district'].nunique() > 1:
    print("\n=== Latency by Health District ===")
    district_stats = df.groupby('health_district')['latency_sec'].agg(['count', 'mean', 'median'])
    print(district_stats.round(2))
    
    # Visualization 11: Small multiples (facet grid)
    g = sns.FacetGrid(df, col='health_district', col_wrap=4, height=3, aspect=1.2)
    g.map(plt.hist, 'latency_sec', bins=20, edgecolor='black', color='steelblue')
    g.set_axis_labels('Latency (seconds)', 'Frequency')
    g.set_titles(col_template="{col_name}")
    g.fig.suptitle('Latency Distribution by Health District', y=1.02, fontsize=16, weight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'latency_by_health_district.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved: latency_by_health_district.png")
else:
    print("\n⚠ health_district column not found or has only one value, skipping this analysis")

## 6. Missing Data Analysis

In [ ]:
# Missing data summary
missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean() * 100).values
}).sort_values('missing_pct', ascending=False)

print("\n=== Missing Data Summary ===")
print(missing_summary[missing_summary['missing_pct'] > 0])

if missing_summary['missing_pct'].sum() == 0:
    print("\n✓ No missing data found!")
else:
    print(f"\n⚠ Total missing values: {missing_summary['missing_count'].sum():,}")

In [ ]:
# Visualization 12: Missing data bar chart
if missing_summary['missing_pct'].max() > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_data = missing_summary[missing_summary['missing_pct'] > 0].sort_values('missing_pct', ascending=True)
    
    missing_data.plot(x='column', y='missing_pct', kind='barh', ax=ax, color='coral', legend=False)
    ax.set_xlabel('Missing Data (%)', fontsize=12)
    ax.set_ylabel('Column', fontsize=12)
    ax.set_title('Missing Data by Column', fontsize=14, weight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'missing_data_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved: missing_data_summary.png")
else:
    print("\n✓ No missing data visualization needed (0% missing)")

## 7. Normality Assessment

Test assumptions for paired t-test.

In [ ]:
# Create baseline comparison (manual baseline = 20 minutes = 1200 seconds)
manual_baseline = 1200  # 20 minutes in seconds
df['manual_latency_est'] = manual_baseline
df['latency_diff'] = df['manual_latency_est'] - df['latency_sec']

print(f"\n=== Baseline Comparison ===")
print(f"Manual Baseline: {manual_baseline} seconds ({manual_baseline/60:.1f} minutes)")
print(f"Automated Mean: {df['latency_sec'].mean():.2f} seconds")
print(f"Mean Difference: {df['latency_diff'].mean():.2f} seconds")
print(f"Std Dev Difference: {df['latency_diff'].std():.2f} seconds")

In [ ]:
# Visualization 13: Q-Q plots for normality
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Q-Q plot for overall difference scores
stats.probplot(df['latency_diff'], dist="norm", plot=axes[0])
axes[0].set_title('Q-Q Plot: Latency Difference Scores (Overall)', fontsize=12, weight='bold')
axes[0].set_xlabel('Theoretical Quantiles', fontsize=11)
axes[0].set_ylabel('Sample Quantiles', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Panel B: Q-Q plot for first data source (if multiple exist)
if df['data_source'].nunique() > 1:
    first_source = df['data_source'].value_counts().index[0]
    source_diff = df[df['data_source'] == first_source]['latency_diff']
    stats.probplot(source_diff, dist="norm", plot=axes[1])
    axes[1].set_title(f'Q-Q Plot: Latency Difference ({first_source.title()})', fontsize=12, weight='bold')
else:
    stats.probplot(df['latency_diff'], dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot: Latency Difference Scores', fontsize=12, weight='bold')

axes[1].set_xlabel('Theoretical Quantiles', fontsize=11)
axes[1].set_ylabel('Sample Quantiles', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'qq_plots_normality.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: qq_plots_normality.png")

In [ ]:
# Shapiro-Wilk test for normality
# Sample if dataset is too large (Shapiro-Wilk has max n=5000)
sample_size = min(5000, len(df['latency_diff']))
sample_diff = df['latency_diff'].sample(sample_size, random_state=12345)

stat, p_value = stats.shapiro(sample_diff)

print(f"\n=== Shapiro-Wilk Test for Normality ===")
print(f"Sample Size: {sample_size:,}")
print(f"Test Statistic: {stat:.6f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n❌ Reject null hypothesis: Data is NOT normally distributed")
    print("   → Consider Wilcoxon signed-rank test as non-parametric alternative")
else:
    print("\n✓ Fail to reject null: Data is consistent with normal distribution")
    print("   → Paired t-test is appropriate")

In [ ]:
# Visualization 14: Distribution with normal overlay
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram
ax.hist(df['latency_diff'], bins=50, density=True, alpha=0.7, 
        edgecolor='black', color='steelblue', label='Observed')

# Fit normal distribution
mu, sigma = df['latency_diff'].mean(), df['latency_diff'].std()
x = np.linspace(df['latency_diff'].min(), df['latency_diff'].max(), 100)
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, 
        label=f'Normal(μ={mu:.1f}, σ={sigma:.1f})')

ax.set_xlabel('Latency Difference (manual - automated, seconds)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Distribution of Difference Scores\nShapiro-Wilk p={p_value:.4f}', 
             fontsize=14, weight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'normality_histogram_overlay.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: normality_histogram_overlay.png")

## 8. Summary and Next Steps

In [ ]:
# Generate summary statistics report
summary = {
    'total_records': len(df),
    'data_sources': df['data_source'].nunique(),
    'latency_mean': df['latency_sec'].mean(),
    'latency_median': df['latency_sec'].median(),
    'latency_p95': df['latency_sec'].quantile(0.95),
    'validation_pass_rate': df['validation_pass_bool'].mean() * 100,
    'missing_data_pct': df.isna().mean().mean() * 100,
    'normality_test_pvalue': p_value,
    'normality_assumption_met': p_value >= 0.05
}

print("\n" + "="*60)
print("EDA SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n" + "="*60)
print("VISUALIZATIONS CREATED")
print("="*60)
figures = list(output_dir.glob('*.png'))
print(f"Total figures saved: {len(figures)}")
for i, fig in enumerate(sorted(figures), 1):
    print(f"  {i}. {fig.name}")

print("\n✓ EDA Complete!")
print("\nNext Steps:")
print("  1. Review visualizations in reports/figures/eda/")
print("  2. Proceed to 02_statistical_analysis.ipynb for hypothesis testing")
print("  3. Document any assumption violations or data quality issues")